# LEAR and ANC interactive inspection

The canonical implementation lives in `lear_model.py` and `run_lear.py`. This notebook is optional and is never executed by `run_full_experiment.py`.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
REPO_ROOT = next(
    candidate for candidate in (start, *start.parents)
    if (candidate / 'experiment_config.py').is_file()
)
sys.path.insert(0, str(REPO_ROOT))

from experiment_config import load_experiment_config
from experiment_manifest import get_lear_run
from pipeline.delivery_index import delivery_clock_index, read_forecast_csv
from pipeline.lear.run_lear import run_anc, run_exaa_naive, run_operational

EXPERIMENT = load_experiment_config(REPO_ROOT)
CONFIG_NAME = 'era5_d56_c1_fundamental'
RUN = get_lear_run(CONFIG_NAME)
RUN

## Optional execution

Change only the guard for the calculation you deliberately want to run. These calls write the same outputs as the command-line entry points.

In [ ]:
RUN_EXAA_NAIVE = False
RUN_OPERATIONAL = False
RUN_ANC = False
ANC_VARIANT = 'fundamental'

if RUN_EXAA_NAIVE:
    run_exaa_naive(EXPERIMENT)
if RUN_OPERATIONAL:
    run_operational(EXPERIMENT, RUN)
if RUN_ANC:
    run_anc(EXPERIMENT, ANC_VARIANT)

## Inspect an existing operational result

In [ ]:
import json
import pandas as pd

result_dir = RUN.output_dir(EXPERIMENT.results_root)
if (result_dir / 'forecast.csv').exists():
    forecast = read_forecast_csv(
        result_dir / 'forecast.csv', required_columns=('y_pred', 'y_true'),
        require_complete_days=True,
    )
    runtime = pd.read_csv(result_dir / 'runtime.csv')
    metrics = pd.read_csv(result_dir / 'metrics.csv')
    metadata = json.loads((result_dir / 'config.json').read_text(encoding='utf-8'))
    display(metadata)
    display(forecast.head())
    display(runtime.head())
    display(metrics)
    plot_forecast = forecast[['y_true', 'y_pred']].iloc[:96 * 7].copy()
    plot_forecast.index = delivery_clock_index(plot_forecast.index)
    plot_forecast.plot(
        figsize=(14, 4), title=CONFIG_NAME, ylabel='EUR/MWh'
    )
else:
    print(f'No existing result at {result_dir}')